# 파일 입력
비전 기능을 갖춘 OpenAI 모델은 PDF 파일을 입력으로 사용할 수도 있습니다. 

모델이 PDF 콘텐츠를 이해할 수 있도록 각 페이지의 추출된 텍스트와 이미지를 모델의 컨텍스트에 추가합니다. 모델은 텍스트와 이미지를 모두 사용하여 응답을 생성할 수 있습니다. 이는 예를 들어 다이어그램에 텍스트에 없는 주요 정보가 포함된 경우 유용합니다.

먼저 Files API를 사용하여 PDF를 업로드한 다음, 모델에 대한 API 요청에서 해당 파일 ID를 참조합니다.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-4.1-nano"

In [4]:
from openai import OpenAI
client = OpenAI()

file = client.files.create(
    file=open("data/삼성중공업-기업리포트.pdf", "rb"),
    purpose="user_data"
)

response = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
                {
                    "type": "input_text",
                    "text": "파일 내에 있는 첫번째 바차트를 설명해줘.",
                },
            ]
        }
    ]
)

print(response.output_text)

파일 내 첫 번째 차트는 "전세계 LNG선 플레이어의 수주잔고와 시장점유율"을 보여주는 그래프입니다. 차트는 두 축으로 구성되어 있는데, 왼쪽 축은 "수주잔고(척)"를 나타내며, 오른쪽 축은 "시장점유율(%)"을 보여줍니다.

그래프는 여러 선으로 구성되어 있는데, 각각의 선은 특정 조선사 또는 회사의 수주잔고와 시장점유율을 표현합니다. 차트 상에서 주요 조선사들이 각각의 수주잔고와 시장점유율을 차지하고 있으며, 전체 LNG선 시장에서 어떤 회사들이 주도권을 가지고 있는지 파악할 수 있습니다.

이 차트는 글로벌 LNG선 시장의 경쟁 구도와 각 회사들의 시장 점유율 변화를 시각적으로 보여줍니다.


### Base64로 인코딩된 파일
PDF 파일 입력도 Base64로 인코딩된 입력으로 보낼 수 있습니다.

In [5]:
import base64

with open("data/삼성중공업-기업리포트.pdf", "rb") as f:
    data = f.read()

base64_string = base64.b64encode(data).decode("utf-8")

response = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": "삼성중공업-기업리포트.pdf",
                    "file_data": f"data:application/pdf;base64,{base64_string}",
                },
                {
                    "type": "input_text",
                    "text": "파일 내에 있는 첫번째 바차트를 설명해줘.",
                },
            ],
        },
    ]
)

print(response.output_text)

첫 번째 차트는 글로벌 LNG선 시장에서 주요 플레이어들의 수주잔고와 시장 점유율(M/S, Market Share)을 보여줍니다. 차트는 좌측 축에 수주잔고(척수, 배)와 우측 축에 시장 점유율(%)가 표시되어 있습니다.

- 수주잔고는 여러 조선소들이 가진 LNG선 주문량을 나타내며, 클락슨 데이터를 기반으로 하고 있습니다.
- 시장 점유율은 각 조선소들이 차지하는 전체 LNG선 수주 비중을 보여줍니다.

차트에 포함된 주요 조선소들은 다음과 같습니다:

- 삼성중공업(목록 상단, 파란색 막대)
- 현대중공업, 대우조선, 삼성중공업, 3자 계약사 등 경쟁사들도 함께 표시되어 있으며, 각각의 수주잔고와 시장 점유율이 나타나 있습니다.

요약하자면, 이 차트는 글로벌 LNG선 시장에서 각 조선소의 현재 수주량과 시장 내 비중을 파악할 수 있도록 도와줍니다. 삼성중공업이 어느 정도의 수주잔고를 확보하고 있으며, 시장 내 점유율이 어떻게 변화하는지를 시각적으로 보여주는 중요한 자료입니다.
